# Simulación Monte Carlo para riesgo de incumplir el plan de producción

## Caso de manufactura

Una planta debe producir una cantidad objetivo de piezas durante un mes. La capacidad real no es fija: cambia por tiempo de ciclo, disponibilidad de máquinas, mantenimiento, paros no planeados, cambios de modelo y scrap.

La pregunta para operaciones es:

**¿Cuál es la probabilidad de cumplir la meta mensual y qué factores explican el riesgo de incumplimiento?**

Este notebook genera miles de escenarios posibles y calcula la producción neta de cada uno.

## Objetivos

- Explicar la simulación Monte Carlo con lenguaje sencillo.
- Modelar incertidumbre operativa de una planta.
- Calcular capacidad bruta y producción neta.
- Estimar probabilidad de cumplir la meta.
- Obtener escenarios optimista, central y desfavorable.
- Calcular una reserva de capacidad.
- Analizar sensibilidad a paros, scrap y tiempo de ciclo.
- Traducir resultados a decisiones de manufactura.

## 1. Técnica y método Monte Carlo

Monte Carlo repite muchas veces un cálculo usando valores aleatorios razonables. Cada repetición representa un posible mes de producción.

### Método

1. Definir la meta de piezas.
2. Identificar factores inciertos.
3. Asignar una distribución a cada factor.
4. Generar un escenario mensual.
5. Calcular horas disponibles, piezas brutas y piezas buenas.
6. Repetir el proceso miles de veces.
7. Resumir la distribución con probabilidades y percentiles.

La simulación no predice un único resultado. Muestra un rango de resultados posibles y la frecuencia con que podrían ocurrir.

## 1.1 Modelo matemático

Horas disponibles:

Horas disponibles = horas programadas - mantenimiento - paros - cambios de modelo

Capacidad bruta:

Piezas brutas = máquinas x horas disponibles x 3,600 / tiempo de ciclo

Producción neta:

Piezas netas = piezas brutas x (1 - porcentaje de scrap)

Cumplimiento:

Cumple = 1 si piezas netas >= meta; en otro caso, 0.

La calidad del resultado depende de los supuestos. En una planta real se deben utilizar históricos por máquina, turno, producto y temporada.

### Comprobación dimensional

horas-máquina × segundos/hora ÷ segundos/pieza = piezas.

El resultado debe ser una cantidad de piezas, no horas ni dinero. Esta revisión evita olvidar el factor 3,600 o mezclar horas de una máquina con la capacidad de toda la planta.

### Lectura sencilla de las unidades

En el cálculo final se usan horas-máquina de toda la planta. Por eso primero se multiplican los días por las horas diarias y por el número de máquinas. Después se restan las horas perdidas de mantenimiento, paro y cambio, que también representan horas-máquina acumuladas de la planta.

La conversión correcta es: horas-máquina × 3,600 segundos por hora ÷ segundos por pieza = piezas. Finalmente, piezas brutas × (1 - proporción de scrap) = piezas buenas.

## 2. Configuración

Usaremos NumPy para generar escenarios, pandas para organizar resultados y Matplotlib y Seaborn para visualizar la distribución de producción.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Entorno listo. Semilla:', RANDOM_STATE)

### Explicación del bloque

La semilla permite que el ejercicio sea repetible. Si otra persona ejecuta el notebook con la misma semilla, obtendrá escenarios comparables. Las gráficas se utilizan para que el resultado sea entendible para producción, calidad y mantenimiento.

## 3. Supuestos operativos

La planta trabaja con dos máquinas durante 26 días, 16 horas por día. La meta mensual es de 120,000 piezas.

Los factores inciertos son:

- Tiempo de ciclo por pieza.
- Horas de mantenimiento programado.
- Horas de paro no planeado.
- Horas de cambios de modelo.
- Porcentaje de scrap.

Los rangos se expresan en segundos, horas y porcentajes realistas para un ejercicio de producción.

### Auditoría de unidades

Para evitar confusiones, el modelo utiliza estas unidades:

| Variable | Unidad | Significado |
|---|---|---|
| dias_laborables | días/mes | días disponibles para producir |
| horas_turno | horas/día | horas programadas por máquina cada día |
| horas_programadas_por_maquina | horas/máquina/mes | días por horas por máquina |
| horas_programadas_totales | horas-máquina/mes | horas por máquina multiplicadas por el número de máquinas |
| tiempo_ciclo | segundos/pieza | tiempo para fabricar una pieza |
| mantenimiento, paros, cambios | horas-máquina/mes | tiempo perdido de la flota completa |
| piezas_brutas | piezas/mes | producción antes de scrap |
| scrap_pct | proporción | porcentaje expresado como decimal entre 0 y 1 |
| piezas_net | piezas buenas/mes | producción liberable después de scrap |

La consistencia es importante: horas-máquina por 3,600 segundos/hora dividido entre segundos/pieza produce piezas. Después se descuenta la proporción de scrap.

In [ ]:
meta_produccion = 140_000
numero_maquinas = 2
dias_laborables = 26
horas_turno = 16
tiempo_ciclo_promedio_s = 18
desviacion_ciclo_s = .7
n_simulaciones = 20_000

mantenimiento_min, mantenimiento_moda, mantenimiento_max = 24, 40, 72
paro_min, paro_moda, paro_max = 8, 24, 70
cambio_min, cambio_moda, cambio_max = 10, 24, 55

print('Meta mensual:', meta_produccion, 'piezas')
print('Horas programadas por máquina:', dias_laborables * horas_turno)
print('Simulaciones:', n_simulaciones)

### Explicación de los supuestos

La meta es la cantidad de piezas buenas que se desea liberar. El tiempo de ciclo es cuánto tarda una máquina en fabricar una pieza. Mantenimiento, paros y cambios reducen las horas disponibles.

Las distribuciones triangulares usan tres valores: mínimo, valor más probable y máximo. Son útiles cuando el equipo conoce rangos operativos aunque no tenga suficientes datos para estimar una distribución compleja.

En este modelo, mantenimiento, paros y cambios se interpretan como horas-máquina acumuladas de toda la planta durante el mes. Por ejemplo, 24 horas pueden significar 12 horas perdidas en cada una de dos máquinas o cualquier combinación equivalente.

## 4. Validación de supuestos

Comprobamos que la meta, las máquinas y los rangos de horas sean válidos antes de ejecutar la simulación.

In [ ]:
horas_programadas_por_maquina = dias_laborables * horas_turno
horas_programadas_totales = numero_maquinas * horas_programadas_por_maquina
assert meta_produccion > 0
assert numero_maquinas > 0
assert horas_programadas_por_maquina > 0
assert tiempo_ciclo_promedio_s > 0
assert 0 < mantenimiento_min <= mantenimiento_moda <= mantenimiento_max < horas_programadas_totales
assert 0 < paro_min <= paro_moda <= paro_max < horas_programadas_totales
assert 0 < cambio_min <= cambio_moda <= cambio_max < horas_programadas_totales
print('Supuestos validados correctamente.')
print('Horas programadas por máquina:', horas_programadas_por_maquina, 'h/mes')
print('Horas programadas totales:', horas_programadas_totales, 'h-máquina/mes')

### Interpretación

Estas comprobaciones evitan simular una planta imposible, por ejemplo con un tiempo de ciclo negativo o más horas de paro que horas programadas. En un proyecto real también se validarían unidades, calendario, disponibilidad de operadores y capacidad máxima de almacenamiento.

## 5. Generación de escenarios mensuales

Cada escenario representa un posible mes. Generamos tiempo de ciclo, mantenimiento, paros, cambios y scrap. Después aplicamos las fórmulas de capacidad.

### Qué significa un escenario

Una fila representa un mes posible. Las horas están expresadas en horas-máquina, el ciclo en segundos por pieza, el scrap como proporción decimal y la producción como piezas buenas por mes. Mantener estas unidades explícitas evita comparar directamente horas con piezas.

In [ ]:
tiempo_ciclo = rng.normal(tiempo_ciclo_promedio_s, desviacion_ciclo_s, n_simulaciones).clip(.1, None)
mantenimiento = rng.triangular(mantenimiento_min, mantenimiento_moda, mantenimiento_max, n_simulaciones)
paros = rng.triangular(paro_min, paro_moda, paro_max, n_simulaciones)
cambios_modelo = rng.triangular(cambio_min, cambio_moda, cambio_max, n_simulaciones)
scrap_pct = rng.beta(3, 97, n_simulaciones)

horas_disponibles = horas_programadas_totales - mantenimiento - paros - cambios_modelo
piezas_brutas = horas_disponibles * 3600 / tiempo_ciclo
piezas_net = piezas_brutas * (1 - scrap_pct)
cumple_meta = piezas_net >= meta_produccion

resultados = pd.DataFrame({
    'tiempo_ciclo_s': tiempo_ciclo,
    'mantenimiento_h': mantenimiento,
    'paros_h': paros,
    'cambios_modelo_h': cambios_modelo,
    'scrap_pct': scrap_pct,
    'horas_disponibles': horas_disponibles,
    'piezas_brutas': piezas_brutas,
    'piezas_net': piezas_net,
    'cumple_meta': cumple_meta
})
print('Escenarios generados:', len(resultados))
display(resultados.head().round(2))

## 5.1 Auditoría de unidades y rangos

Este bloque comprueba que las horas disponibles, el tiempo de ciclo, el scrap y las piezas netas tengan valores físicamente válidos.

In [ ]:
auditoria_unidades = {
    'horas_por_maquina_h_mes': horas_programadas_por_maquina,
    'horas_totales_h_maquina_mes': horas_programadas_totales,
    'horas_disponibles_min_h_maquina': round(float(horas_disponibles.min()), 2),
    'tiempo_ciclo_min_s_pieza': round(float(tiempo_ciclo.min()), 2),
    'tiempo_ciclo_max_s_pieza': round(float(tiempo_ciclo.max()), 2),
    'scrap_min_proporcion': round(float(scrap_pct.min()), 4),
    'scrap_max_proporcion': round(float(scrap_pct.max()), 4),
    'piezas_net_min': round(float(piezas_net.min())),
    'piezas_net_max': round(float(piezas_net.max()))
}
print(auditoria_unidades)
assert (horas_disponibles > 0).all()
assert (tiempo_ciclo > 0).all()
assert ((scrap_pct >= 0) & (scrap_pct <= 1)).all()
assert (piezas_net >= 0).all()
print('Auditoría de unidades: OK')

### Explicación detallada del cálculo

Cada columna representa un factor incierto. El tiempo de ciclo se genera alrededor de su promedio. El scrap usa una distribución beta para producir porcentajes pequeños y positivos.

Las horas disponibles se obtienen restando pérdidas a las horas programadas. Las piezas brutas son la capacidad antes de scrap. Las piezas netas son las piezas buenas disponibles después de descontar defectos. Finalmente, cumple_meta indica si el escenario alcanza las 120,000 piezas.

## 6. Revisión de la simulación

Revisamos promedios, mínimos, máximos y correlaciones básicas para confirmar que la producción neta y las pérdidas están en rangos razonables.

In [ ]:
display(resultados[['tiempo_ciclo_s','mantenimiento_h','paros_h','cambios_modelo_h','scrap_pct','horas_disponibles','piezas_net']].describe().round(2))
print('Probabilidad inicial de cumplimiento:', f'{resultados.cumple_meta.mean():.1%}')

### Interpretación

El resumen permite identificar si la variación está dominada por disponibilidad, velocidad o calidad. Si las horas disponibles caen mucho, mantenimiento y paros podrían ser los principales riesgos. Si las horas son estables pero piezas netas varía, el tiempo de ciclo o scrap merece mayor atención.

## 7. Distribución de producción neta

La gráfica muestra cuántas simulaciones producen ciertos volúmenes de piezas buenas. La línea vertical representa la meta.

### Comprobación de realidad operativa

La producción neta no es la capacidad teórica: ya descuenta mantenimiento, paros, cambios y scrap. Por eso es la cifra que debe compararse contra la meta de piezas buenas liberables.

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(resultados.piezas_net, bins=60, kde=True, color='#4C78A8')
plt.axvline(meta_produccion, color='#D62728', linestyle='--', linewidth=2, label='Meta')
plt.axvline(resultados.piezas_net.mean(), color='#2CA02C', linestyle='--', linewidth=2, label='Media')
plt.title('Distribución simulada de piezas netas')
plt.xlabel('Piezas buenas producidas')
plt.ylabel('Número de escenarios')
plt.legend()
plt.show()

### Interpretación de la distribución

Las simulaciones a la derecha de la meta cumplen el plan. Las que quedan a la izquierda representan incumplimiento. La distancia entre la media y la meta muestra si el plan tiene holgura o si depende de que todo salga cerca del escenario esperado.

## 8. Probabilidad de cumplimiento

Calculamos la proporción de escenarios que alcanza la meta y la proporción que queda por debajo.

In [ ]:
probabilidad_cumplimiento = resultados.cumple_meta.mean()
probabilidad_incumplimiento = 1 - probabilidad_cumplimiento
print(f'Probabilidad de cumplir la meta: {probabilidad_cumplimiento:.1%}')
print(f'Probabilidad de incumplir: {probabilidad_incumplimiento:.1%}')
print(f'Producción media: {resultados.piezas_net.mean():,.0f} piezas')
print(f'Desviación estándar: {resultados.piezas_net.std():,.0f} piezas')

### Interpretación ejecutiva

Si la probabilidad de cumplimiento es 80%, significa que 8 de cada 10 escenarios simulados llegan a la meta bajo estos supuestos. No significa que la planta vaya a cumplir exactamente 80% de las veces; es una estimación del riesgo.

Una probabilidad baja puede requerir mantenimiento preventivo, reducción de cambios, mejora de scrap, capacidad adicional u horas extra.

## 9. Percentiles y reserva de capacidad

Los percentiles describen niveles de producción. El percentil 5 es una referencia de escenario desfavorable: 95% de los escenarios producen más que ese valor.

In [ ]:
percentiles = [5, 25, 50, 75, 95]
tabla_percentiles = pd.DataFrame({
    'percentil': percentiles,
    'piezas_net': [np.percentile(resultados.piezas_net, p) for p in percentiles]
})
tabla_percentiles['brecha_vs_meta'] = tabla_percentiles.piezas_net - meta_produccion
display(tabla_percentiles.style.format({'piezas_net':'{:,.0f}', 'brecha_vs_meta':'{:,.0f}'}))
reserva_capacidad = meta_produccion - np.percentile(resultados.piezas_net, 5)
print(f'Capacidad adicional para cubrir el percentil 5: {max(0, reserva_capacidad):,.0f} piezas')

### Interpretación detallada

El percentil 50 representa el escenario central. El percentil 5 muestra una producción baja pero posible. Si está debajo de la meta, la diferencia indica cuántas piezas podrían faltar en un mes desfavorable.

La reserva puede cubrirse con horas extra, una máquina alterna, inventario de seguridad o reducción de tiempos de cambio. No necesariamente significa producir esa cantidad adicional todos los meses.

## 10. Escenarios operativos

Convertimos la distribución en escenarios fáciles de comunicar a dirección y operaciones.

In [ ]:
escenarios = pd.DataFrame({
    'escenario': ['Desfavorable', 'Central', 'Favorable'],
    'percentil': [5, 50, 95]
})
escenarios['piezas_net'] = escenarios.percentil.apply(lambda p: np.percentile(resultados.piezas_net, p))
escenarios['cumplimiento_meta_pct'] = 100 * escenarios.piezas_net / meta_produccion
display(escenarios.style.format({'piezas_net':'{:,.0f}', 'cumplimiento_meta_pct':'{:.1f}%'}))

### Interpretación

El escenario desfavorable ayuda a preparar acciones preventivas. El central representa el resultado típico. El favorable muestra la capacidad que podría alcanzarse cuando los tiempos de pérdida son bajos.

Estos escenarios no sustituyen el programa maestro; sirven para evaluar qué tan robusto es el plan ante variaciones normales.

## 11. Sensibilidad de factores

Cambiamos un factor a la vez para estimar qué acciones tendrían mayor impacto en el riesgo de incumplimiento.

In [ ]:
def simular_produccion(ciclo_promedio, paro_moda, scrap_alpha):
    ciclo = rng.normal(ciclo_promedio, desviacion_ciclo_s, n_simulaciones).clip(.1, None)
    mant = rng.triangular(mantenimiento_min, mantenimiento_moda, mantenimiento_max, n_simulaciones)
    paro = rng.triangular(paro_min, paro_moda, paro_max, n_simulaciones)
    cambio = rng.triangular(cambio_min, cambio_moda, cambio_max, n_simulaciones)
    scrap = rng.beta(scrap_alpha, 100-scrap_alpha, n_simulaciones)
    horas = horas_programadas_totales - mant - paro - cambio
    netas = horas * 3600 / ciclo * (1-scrap)
    return netas

sensibilidad_filas = []
for nombre, ciclo_param, paro_param, scrap_param in [
    ('Base', 18, 24, 3),
    ('Ciclo +5%', 18.9, 24, 3),
    ('Paros -20%', 18, 19, 3),
    ('Scrap +50%', 18, 24, 4.5)
]:
    netas = simular_produccion(ciclo_param, paro_param, scrap_param)
    sensibilidad_filas.append({
        'escenario': nombre,
        'prob_cumplimiento': (netas >= meta_produccion).mean(),
        'produccion_p5': np.percentile(netas, 5),
        'produccion_media': np.mean(netas)
    })
sensibilidad = pd.DataFrame(sensibilidad_filas)
display(sensibilidad.style.format({
    'prob_cumplimiento':'{:.1%}',
    'produccion_p5':'{:,.0f}',
    'produccion_media':'{:,.0f}'
}))

### Interpretación de sensibilidad

Ciclo +5% representa una pérdida de velocidad. Paros -20% representa una mejora de disponibilidad. Scrap +50% muestra el impacto de calidad. La comparación permite decidir si conviene invertir primero en mantenimiento, reducción de setup, mejora de proceso o reducción de defectos.

La acción prioritaria debe ser la que reduzca riesgo con un costo razonable y sea operativamente viable.

## 12. Conclusiones

- Monte Carlo convierte incertidumbre de producción en una distribución de resultados.
- La probabilidad de cumplimiento comunica el riesgo mejor que una capacidad promedio aislada.
- El percentil 5 ayuda a dimensionar una reserva de capacidad para meses desfavorables.
- Tiempo de ciclo, paros, cambios y scrap afectan directamente la producción neta.
- La sensibilidad ayuda a priorizar acciones de mantenimiento, calidad y mejora continua.
- Los parámetros deben actualizarse con datos reales por máquina, turno, producto y temporada.

### Plan recomendado

1. Reemplazar supuestos con históricos de producción, downtime y scrap.
2. Separar paros planeados y no planeados.
3. Modelar cambios de modelo por familia de producto.
4. Validar la simulación contra meses pasados.
5. Revisar mensualmente la probabilidad de cumplimiento y la reserva de capacidad.

## 13. Resumen reproducible

In [ ]:
print({
    'simulaciones': n_simulaciones,
    'meta_piezas': meta_produccion,
    'probabilidad_cumplimiento': round(float(probabilidad_cumplimiento),3),
    'produccion_media': round(float(resultados.piezas_net.mean())),
    'percentil_5': round(float(np.percentile(resultados.piezas_net,5))),
    'reserva_capacidad_piezas': round(float(max(0, reserva_capacidad)))
})